# Notebook 02 — Fraud Detection Platform: Validation, Robustness & Deployment Readiness
**Master Playbook Sections 5, 7, 9, 11, 17.4, 19 — real Two-Gate validation, drift-monitoring baseline, adversarial-robustness test, and model card, built on Notebook 01's real, already-validated champion model (CatBoost) and its real measured results — no retraining.**


In [ ]:
# ============================================================
# SETUP -- WARP-optimized environment (thread ceiling set BEFORE any ML import)
# CPU/RAM thresholds (user-specified): CPU 90-95%, RAM 90%.
# ============================================================
import os, time, json, pickle, warnings, subprocess, sys
warnings.filterwarnings("ignore")

CPU_THRESHOLD_PCT = 93   # midpoint of the requested 90-95% band
RAM_THRESHOLD_PCT = 90

_N_THREADS = max(1, int((os.cpu_count() or 4) * (CPU_THRESHOLD_PCT / 100) // 1))
os.environ.setdefault("OMP_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(_N_THREADS))
os.environ.setdefault("POLARS_MAX_THREADS", str(_N_THREADS))

# Auto-install guard -- same self-contained pattern as Notebook 01, extended
# to every package this notebook needs (including catboost/scipy, in case
# this is run in a fresh environment that only has Notebook 01's deps
# partially installed).
for _pkg in ("polars", "psutil", "pyarrow", "scipy", "catboost"):
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", _pkg], check=True)

import polars as pl
import psutil
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp  # noqa: F401 -- import-order sanity check before the module bootstrap needs it

# display() is a Jupyter/IPython builtin -- guaranteed in a notebook kernel,
# but not if this cell is ever run via a plain script or nbconvert without
# a display hook. Defensive fallback so this notebook never breaks on that.
try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

_ram_start = psutil.virtual_memory()
print(f"WARP thread ceiling: {_N_THREADS} threads (of {os.cpu_count()} available cores, target {CPU_THRESHOLD_PCT}%)")
print(f"RAM at startup: {_ram_start.percent:.1f}% used ({_ram_start.used/1e9:.2f} GB / {_ram_start.total/1e9:.2f} GB) -- target ceiling {RAM_THRESHOLD_PCT}%")
if _ram_start.percent >= RAM_THRESHOLD_PCT:
    print(f"WARNING: RAM already at/above the {RAM_THRESHOLD_PCT}% target before this notebook has loaded any data.")
print("Note: Windows has no simple pure-Python API for a hard OS-level RAM cap "
      "(that needs native Job Object APIs via pywin32). This notebook reports "
      "live usage and reduces real work via vectorization instead of pretending "
      "to enforce a hard limit it cannot actually guarantee.")
print("Setup complete.")

##############################################################################
# REPO-LAYOUT BOOTSTRAP -- verified detection (Notebook 01's fix, reused
# unchanged). Redirects this run's outputs into reports/nb2_results/ and
# locates Notebook 01's real outputs in the sibling reports/nb1_results/.
##############################################################################
_KNOWN_REPO_ROOT = r"C:\Users\rnand\Downloads\Fraud_Detection_Platform\Fraud_Detection_Platform_repo_only"

def _looks_like_repo(_p):
    return os.path.isdir(os.path.join(_p, "notebooks")) or os.path.exists(os.path.join(_p, "requirements.txt"))

_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, ".."))

if os.path.isdir(_KNOWN_REPO_ROOT):
    REPO_ROOT = _KNOWN_REPO_ROOT
elif _looks_like_repo(_parent):
    REPO_ROOT = _parent
elif _looks_like_repo(_cwd):
    REPO_ROOT = _cwd
else:
    REPO_ROOT = _cwd

_SCAFFOLD_DIRS = [
    "data/raw", "data/processed", "notebooks/starters",
    "src", "deployment", "reports", "docs", "publish_drafts", "tests",
]
for _rel in _SCAFFOLD_DIRS:
    try:
        os.makedirs(os.path.join(REPO_ROOT, *_rel.split("/")), exist_ok=True)
    except PermissionError as _e:
        print(f"WARNING: could not create '{_rel}' under {REPO_ROOT} ({_e}). Skipping.")

NB1_RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb1_results")
RESULTS_DIR = os.path.join(REPO_ROOT, "reports", "nb2_results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
try:
    os.makedirs(FIG_DIR, exist_ok=True)
except PermissionError:
    RESULTS_DIR = os.path.join(_cwd, "nb2_results")
    FIG_DIR = os.path.join(RESULTS_DIR, "figures")
    os.makedirs(FIG_DIR, exist_ok=True)
    print(f"WARNING: falling back to {RESULTS_DIR} (no write access to {REPO_ROOT}).")

print(f"Repo root:      {REPO_ROOT}")
print(f"NB1 results in: {NB1_RESULTS_DIR}")
print(f"NB2 results in: {RESULTS_DIR}")

##############################################################################
# SELF-CONTAINED MODULE BOOTSTRAP -- writes the four real governance/
# validation modules to disk (byte-identical to src/, encoding="utf-8"
# pinned explicitly, unconditional overwrite -- no read-back-before-write)
# so this notebook has zero external .py dependencies, same pattern as
# Notebook 01.
##############################################################################
from pathlib import Path as _Path

_HERE = _Path.cwd()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

_MODULE_SOURCES = {
    "two_gate_validation.py": '"""\ntwo_gate_validation.py\nReusable module — implements Section 7 of the Master Playbook: the two-gate\nvalidation architecture, extended with the Stress-Test Scenario sub-gate.\n\nGate 1 = structural integrity (always computable; passing does not mean the\nmodel is good, only that the pipeline ran correctly).\nGate 2 = statistical robustness + concentration + stress-test scenario\n(the gate that can actually fail and should block promotion).\n\nThis module reports REAL numbers only — it never asserts a verdict for you;\nit returns the evidence and a pass/fail against thresholds YOU set explicitly,\nso no threshold is silently assumed.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom dataclasses import dataclass, field\n\n\n# ---------------------------------------------------------------------------\n# GATE 1 — Structural Integrity\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Gate1Result:\n    checks: dict[str, bool] = field(default_factory=dict)\n\n    @property\n    def all_passed(self) -> bool:\n        return all(self.checks.values())\n\n\ndef run_gate1_structural_checks(df: pd.DataFrame, required_columns: list[str],\n                                 label_column: str) -> Gate1Result:\n    result = Gate1Result()\n\n    result.checks["schema_has_required_columns"] = all(\n        c in df.columns for c in required_columns\n    )\n    result.checks["no_null_in_label"] = df[label_column].isna().sum() == 0\n    result.checks["label_is_binary"] = set(df[label_column].unique()).issubset({0, 1})\n    result.checks["row_count_positive"] = len(df) > 0\n    # No-leakage sanity check: label must not equal a trivial function of\n    # itself once cast (catches an accidental duplicate/label-copy column).\n    numeric_cols = df.select_dtypes(include=[np.number]).columns\n    suspicious_leak_cols = [\n        c for c in numeric_cols\n        if c != label_column and df[c].corr(df[label_column]) > 0.999\n    ]\n    result.checks["no_perfect_correlation_leakage"] = len(suspicious_leak_cols) == 0\n    if suspicious_leak_cols:\n        result.checks["_leak_columns_found"] = suspicious_leak_cols  # diagnostic, not boolean\n\n    return result\n\n\n# ---------------------------------------------------------------------------\n# GATE 2 — Statistical Robustness / Concentration / Stress-Test\n# ---------------------------------------------------------------------------\n\n@dataclass\nclass Gate2Result:\n    cv_auc_scores: list[float]\n    cv_pr_auc_scores: list[float]\n    concentration_report: pd.DataFrame\n    stress_test_report: dict\n\n    @property\n    def cv_stability_ok(self, max_std: float = 0.03) -> bool:\n        return float(np.std(self.cv_pr_auc_scores)) <= max_std\n\n\ndef concentration_report_by_amount_and_time(\n    df: pd.DataFrame, amount_col: str, time_col: str,\n    y_true_col: str, y_pred_col: str, n_amount_bands: int = 5,\n) -> pd.DataFrame:\n    """\n    Real segment-cut breakdown replacing the demographic cuts unavailable on\n    this anonymized dataset (Section 2\'s disclosed limitation) — false-positive\n    and false-negative rates broken out by Amount band and hour-of-day, so\n    concentration risk is checked on the real segments that ARE available.\n    """\n    work = df.copy()\n    work["_amount_band"] = pd.qcut(work[amount_col], n_amount_bands, duplicates="drop")\n    work["_hour_of_day"] = (work[time_col] // 3600) % 24\n\n    rows = []\n    for (band, hour), g in work.groupby(["_amount_band", "_hour_of_day"], observed=True):\n        fp = ((g[y_true_col] == 0) & (g[y_pred_col] == 1)).sum()\n        fn = ((g[y_true_col] == 1) & (g[y_pred_col] == 0)).sum()\n        n = len(g)\n        rows.append({\n            "amount_band": str(band), "hour_of_day": int(hour), "n": n,\n            "false_positive_rate": fp / n if n else np.nan,\n            "false_negative_rate": fn / n if n else np.nan,\n        })\n    return pd.DataFrame(rows)\n\n\ndef stress_test_scenario(\n    df: pd.DataFrame, amount_col: str, scenario_volume_multiplier: float,\n    scenario_fraud_rate_multiplier: float, base_fraud_rate: float,\n) -> dict:\n    """\n    Real, disclosed hypothetical scenario per Section 7/19: since the exact\n    confidential Fed severely-adverse scenario file is not available to a\n    portfolio project, this applies a documented, labeled hypothetical shock\n    (e.g., 2x transaction volume, 3x fraud rate during a stress event) and\n    projects the resulting loss impact on THIS dataset\'s real Amount\n    distribution. Every multiplier is a stated ASSUMPTION, never invented\n    silently — pass real historical-analogue multipliers if you have them.\n    """\n    projected_fraud_rate = base_fraud_rate * scenario_fraud_rate_multiplier\n    projected_transaction_count = len(df) * scenario_volume_multiplier\n    avg_amount = df[amount_col].mean()\n\n    projected_fraud_count = projected_transaction_count * projected_fraud_rate\n    projected_loss = projected_fraud_count * avg_amount\n\n    return {\n        "ASSUMPTION_scenario_volume_multiplier": scenario_volume_multiplier,\n        "ASSUMPTION_scenario_fraud_rate_multiplier": scenario_fraud_rate_multiplier,\n        "base_fraud_rate_real": base_fraud_rate,\n        "projected_fraud_rate_under_scenario": projected_fraud_rate,\n        "projected_transaction_count": projected_transaction_count,\n        "projected_fraud_loss_usd_or_eur": float(projected_loss),\n    }\n',
    "drift_monitoring.py": '"""\ndrift_monitoring.py\nReusable module — implements Section 9 of the Master Playbook: PSI/CSI/KS\ndrift computation with the Tier 1 alert thresholds from Section 9\'s table.\n\nThis is the module that turns "we designed a monitoring plan" (v1-v6 of the\nplaybook) into something you actually run, repeatedly, over real elapsed\ntime (Section 21 Phase 8) to build a genuine drift history.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom dataclasses import dataclass\nfrom scipy.stats import ks_2samp\n\n# Tier 1 alert thresholds — from Section 9 of the Master Playbook.\n# Change these only with a written justification, per Section 3\'s tiering rule.\nTIER_THRESHOLDS = {\n    1: {"psi_alert": 0.25, "ks_pvalue_alert": 0.01, "pr_auc_drop_alert": 0.05},\n    2: {"psi_alert": 0.35, "ks_pvalue_alert": 0.005, "pr_auc_drop_alert": 0.08},\n    3: {"psi_alert": 0.50, "ks_pvalue_alert": 0.001, "pr_auc_drop_alert": 0.12},\n}\n\n\n@dataclass\nclass DriftReport:\n    feature_psi: dict\n    score_ks_statistic: float\n    score_ks_pvalue: float\n    pr_auc_baseline: float\n    pr_auc_current: float\n    tier: int\n\n    @property\n    def pr_auc_drop(self) -> float:\n        return self.pr_auc_baseline - self.pr_auc_current\n\n    @property\n    def alerts(self) -> dict[str, bool]:\n        t = TIER_THRESHOLDS[self.tier]\n        return {\n            "feature_drift_alert": any(v > t["psi_alert"] for v in self.feature_psi.values()),\n            "score_drift_alert": self.score_ks_pvalue < t["ks_pvalue_alert"],\n            "performance_drift_alert": self.pr_auc_drop > t["pr_auc_drop_alert"],\n        }\n\n    @property\n    def any_alert(self) -> bool:\n        return any(self.alerts.values())\n\n\ndef population_stability_index(expected: np.ndarray, actual: np.ndarray,\n                                n_bins: int = 10) -> float:\n    """\n    Real PSI computation between a training-time (\'expected\') and current\n    (\'actual\') distribution of one feature. PSI > 0.25 is the Tier 1 alert\n    threshold from Section 9 — this function computes the real number, it\n    does not assume a verdict.\n    """\n    breakpoints = np.linspace(0, 100, n_bins + 1)\n    bin_edges = np.percentile(expected, breakpoints)\n    bin_edges[0], bin_edges[-1] = -np.inf, np.inf\n\n    expected_pct = np.histogram(expected, bins=bin_edges)[0] / len(expected)\n    actual_pct = np.histogram(actual, bins=bin_edges)[0] / len(actual)\n\n    # Avoid division by zero / log(0) on empty bins with a small floor.\n    expected_pct = np.where(expected_pct == 0, 1e-6, expected_pct)\n    actual_pct = np.where(actual_pct == 0, 1e-6, actual_pct)\n\n    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))\n    return float(psi)\n\n\ndef compute_drift_report(\n    training_features: pd.DataFrame,\n    current_features: pd.DataFrame,\n    training_scores: np.ndarray,\n    current_scores: np.ndarray,\n    pr_auc_baseline: float,\n    pr_auc_current: float,\n    top_features: list[str],\n    tier: int = 1,\n) -> DriftReport:\n    """\n    Runs the real PSI (per top-SHAP feature), KS (on the score distribution),\n    and performance-drift (PR-AUC delta) checks in one call, and evaluates\n    them against the Section 9 Tier thresholds.\n\n    top_features should be the real top-10 SHAP features from Section 6\'s\n    explainability step — drift on features the model doesn\'t actually rely\n    on matters far less than drift on the ones driving its decisions.\n    """\n    feature_psi = {\n        feat: population_stability_index(\n            training_features[feat].values, current_features[feat].values\n        )\n        for feat in top_features\n    }\n\n    ks_stat, ks_pvalue = ks_2samp(training_scores, current_scores)\n\n    return DriftReport(\n        feature_psi=feature_psi,\n        score_ks_statistic=float(ks_stat),\n        score_ks_pvalue=float(ks_pvalue),\n        pr_auc_baseline=pr_auc_baseline,\n        pr_auc_current=pr_auc_current,\n        tier=tier,\n    )\n\n\ndef early_vs_late_window_proxy(df: pd.DataFrame, time_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """\n    Section 9\'s documented proxy for this dataset\'s real limitation: only a\n    ~48-hour window exists, so a genuine multi-period drift run isn\'t\n    possible. This splits the real Time column into early vs. late halves\n    as the disclosed structural stand-in until real multi-period data exists\n    (Section 21 Phase 8).\n    """\n    midpoint = df[time_col].median()\n    early = df[df[time_col] <= midpoint]\n    late = df[df[time_col] > midpoint]\n    return early, late\n',
    "adversarial_robustness.py": '"""\nadversarial_robustness.py\nReal, runnable adversarial-robustness test for the Fraud Detection Platform\n— Master Playbook Section 19 (360-degree gap), Phase 3 of the Section 21\nUltra-Powerful Execution Roadmap.\n\nWHY THIS EXISTS (real grounding, not invented):\nFraud detection is an adversarially contested domain — unlike most retail\ncredit-risk models, the "bad actor" here actively adapts to evade detection.\nChen et al., "Cost-Aware Robust Tree Ensembles for Security Applications"\n(USENIX Security 2021, building on earlier RAID work on adversarial evasion\nof fraud/intrusion classifiers) show that attackers who know or can probe a\nmodel\'s decision boundary will strategically adjust the features cheapest\nfor them to change — for a card-transaction model, that is overwhelmingly\nthe transaction Amount (split a large fraudulent charge into several small\nones) and, to a lesser extent, the timing/Time of the transaction (route\nfraud through hours when the model has seen less fraud historically).\n\nThis module does NOT claim your model is or is not robust — it gives you\nthe exact, real test to run against YOUR trained model and YOUR real\nholdout data, and prints real numbers. Per the standing execution-boundary\nrule, it is written but not executed here.\n\nTwo real tests are implemented:\n  1. perturbation_sensitivity_test — for a grid of Amount/Time perturbations,\n     what fraction of TRUE FRAUD cases that the model currently catches\n     would it miss after the perturbation? (a coarse, fast sweep)\n  2. boundary_search_attack — for each true-fraud case the model currently\n     catches, greedily shrink Amount in small steps until the model\'s score\n     drops below the operating threshold (or a max iteration/step budget is\n     hit). Reports the real % of caught fraud that is "evadable" within a\n     bounded, realistic amount change, and the median amount-change needed.\n     This is the closest real analogue, on this dataset, to a black-box\n     evasion attack a fraud ring could mount by structuring transactions.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Callable, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nRANDOM_SEED = 42\n\n\n@dataclass\nclass PerturbationSensitivityResult:\n    perturbation_type: str          # "amount" or "time"\n    perturbation_value: float       # e.g. -0.20 for a 20% amount cut, or +6 for a 6-hour time shift\n    n_fraud_originally_caught: int\n    n_fraud_still_caught_after: int\n    fraud_evasion_rate: float       # (originally_caught - still_caught) / originally_caught\n\n\n@dataclass\nclass BoundarySearchResult:\n    n_fraud_cases_tested: int\n    n_evadable_within_budget: int\n    evasion_rate_within_budget: float\n    median_amount_change_pct_for_evasion: float | None\n    step_pct: float\n    max_amount_change_pct_budget: float\n    per_case_detail: pd.DataFrame = field(repr=False)\n\n\ndef perturbation_sensitivity_test(\n    score_fn: Callable[[pd.DataFrame], np.ndarray],\n    X_fraud: pd.DataFrame,\n    threshold: float,\n    amount_col: str = "Amount",\n    time_col: str = "Time",\n    amount_fractions: Sequence[float] = (-0.5, -0.3, -0.2, -0.1, 0.1, 0.2, 0.3, 0.5),\n    time_shift_hours: Sequence[float] = (-6, -3, 3, 6, 12),\n    seconds_per_hour: float = 3600.0,\n) -> list[PerturbationSensitivityResult]:\n    """\n    score_fn: a callable that takes a DataFrame of the model\'s real input\n    features and returns real fraud-probability scores (e.g.\n    lambda X: champion_model.predict_proba(X)[:, 1]).\n    X_fraud: the REAL feature rows for transactions with Class == 1 that the\n    model currently scores >= threshold (i.e., the fraud it currently catches).\n    threshold: the REAL operating threshold chosen via\n    class_imbalance_utils.best_threshold_by_cost.\n\n    Returns one result per perturbation tested. All numbers are computed\n    from whatever X_fraud / score_fn you pass in — nothing here is assumed.\n    """\n    if len(X_fraud) == 0:\n        raise ValueError("X_fraud is empty — pass the real fraud rows the model currently catches.")\n\n    base_scores = score_fn(X_fraud)\n    originally_caught_mask = base_scores >= threshold\n    n_originally_caught = int(originally_caught_mask.sum())\n    if n_originally_caught == 0:\n        raise ValueError(\n            "None of the rows in X_fraud score >= threshold with the given score_fn/threshold — "\n            "check that X_fraud really is the currently-caught fraud subset."\n        )\n\n    results: list[PerturbationSensitivityResult] = []\n\n    for frac in amount_fractions:\n        X_pert = X_fraud.copy()\n        X_pert.loc[originally_caught_mask, amount_col] = (\n            X_pert.loc[originally_caught_mask, amount_col] * (1.0 + frac)\n        ).clip(lower=0.0)\n        new_scores = score_fn(X_pert)\n        still_caught = int((new_scores[originally_caught_mask.values] >= threshold).sum()) \\\n            if hasattr(originally_caught_mask, "values") else int((new_scores[originally_caught_mask] >= threshold).sum())\n        evasion_rate = (n_originally_caught - still_caught) / n_originally_caught\n        results.append(PerturbationSensitivityResult(\n            "amount", frac, n_originally_caught, still_caught, evasion_rate\n        ))\n\n    for hrs in time_shift_hours:\n        X_pert = X_fraud.copy()\n        X_pert.loc[originally_caught_mask, time_col] = (\n            X_pert.loc[originally_caught_mask, time_col] + hrs * seconds_per_hour\n        ).clip(lower=0.0)\n        new_scores = score_fn(X_pert)\n        still_caught = int((new_scores[originally_caught_mask.values] >= threshold).sum()) \\\n            if hasattr(originally_caught_mask, "values") else int((new_scores[originally_caught_mask] >= threshold).sum())\n        evasion_rate = (n_originally_caught - still_caught) / n_originally_caught\n        results.append(PerturbationSensitivityResult(\n            "time", hrs, n_originally_caught, still_caught, evasion_rate\n        ))\n\n    return results\n\n\ndef boundary_search_attack(\n    score_fn: Callable[[pd.DataFrame], np.ndarray],\n    X_fraud: pd.DataFrame,\n    threshold: float,\n    amount_col: str = "Amount",\n    step_pct: float = 0.02,\n    max_amount_change_pct_budget: float = 0.90,\n) -> BoundarySearchResult:\n    """\n    Greedy, real, black-box-style evasion search: for every fraud case the\n    model currently catches, shrink Amount by step_pct at a time (a fraud\n    ring\'s cheapest, most realistic lever — splitting/structuring a charge)\n    until the score drops below threshold or the cumulative reduction would\n    exceed max_amount_change_pct_budget (default: won\'t credit an "evasion"\n    that requires cutting the charge by more than 90%, since that mostly\n    defeats the fraud\'s own purpose).\n\n    This gives one real, honest number: what fraction of currently-caught\n    fraud can be evaded within a realistic amount-shrinkage budget, and how\n    much shrinkage that typically takes. Nothing here is estimated — it is\n    computed by actually re-scoring the perturbed rows with your real model.\n    """\n    base_scores = score_fn(X_fraud)\n    caught = X_fraud[base_scores >= threshold].copy()\n    n_tested = len(caught)\n    if n_tested == 0:\n        return BoundarySearchResult(0, 0, 0.0, None, step_pct, max_amount_change_pct_budget,\n                                     pd.DataFrame(columns=["evaded", "amount_change_pct_used"]))\n\n    detail_rows = []\n    for idx, row in caught.iterrows():\n        row_df = pd.DataFrame([row])\n        cum_reduction = 0.0\n        evaded = False\n        while cum_reduction < max_amount_change_pct_budget:\n            cum_reduction += step_pct\n            trial = row_df.copy()\n            trial[amount_col] = trial[amount_col] * (1.0 - cum_reduction)\n            trial[amount_col] = trial[amount_col].clip(lower=0.0)\n            score = score_fn(trial)[0]\n            if score < threshold:\n                evaded = True\n                break\n        detail_rows.append({\n            "index": idx,\n            "evaded": evaded,\n            "amount_change_pct_used": cum_reduction if evaded else None,\n        })\n\n    detail_df = pd.DataFrame(detail_rows)\n    n_evadable = int(detail_df["evaded"].sum())\n    evasion_rate = n_evadable / n_tested\n    median_change = (\n        float(detail_df.loc[detail_df["evaded"], "amount_change_pct_used"].median())\n        if n_evadable > 0 else None\n    )\n\n    return BoundarySearchResult(\n        n_fraud_cases_tested=n_tested,\n        n_evadable_within_budget=n_evadable,\n        evasion_rate_within_budget=evasion_rate,\n        median_amount_change_pct_for_evasion=median_change,\n        step_pct=step_pct,\n        max_amount_change_pct_budget=max_amount_change_pct_budget,\n        per_case_detail=detail_df,\n    )\n\n\ndef print_report(sens_results: list[PerturbationSensitivityResult], boundary_result: BoundarySearchResult) -> None:\n    """Prints the real results in a form you can paste directly into Section 19 as evidence."""\n    print("=" * 70)\n    print("ADVERSARIAL ROBUSTNESS — PERTURBATION SENSITIVITY (real, from your model)")\n    print("=" * 70)\n    for r in sens_results:\n        print(f"  [{r.perturbation_type:6s} {r.perturbation_value:+.2f}] "\n              f"caught before={r.n_fraud_originally_caught}, "\n              f"caught after={r.n_fraud_still_caught_after}, "\n              f"evasion_rate={r.fraud_evasion_rate:.2%}")\n\n    print("=" * 70)\n    print("ADVERSARIAL ROBUSTNESS — GREEDY BOUNDARY-SEARCH ATTACK (real, from your model)")\n    print("=" * 70)\n    print(f"  Fraud cases tested (currently caught): {boundary_result.n_fraud_cases_tested}")\n    print(f"  Evadable within {boundary_result.max_amount_change_pct_budget:.0%} amount-cut budget: "\n          f"{boundary_result.n_evadable_within_budget} "\n          f"({boundary_result.evasion_rate_within_budget:.2%})")\n    if boundary_result.median_amount_change_pct_for_evasion is not None:\n        print(f"  Median amount-cut needed to evade: "\n              f"{boundary_result.median_amount_change_pct_for_evasion:.2%}")\n    else:\n        print("  No evadable cases found within budget.")\n    print("\\nThis is your real, honest adversarial-robustness evidence for Section 19 — "\n          "not an estimate.")\n',
    "model_card_generator.py": '"""\nmodel_card_generator.py\nReusable module — implements Section 11 of the Master Playbook: the\nstandardized, version-locked Model Card required alongside the financial-\nimpact package for every champion model.\n\nPopulate ModelCardData from your OWN real run\'s results, then call\nrender_markdown() / render_html() — this module formats real data you\nprovide; it never invents a metric.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone\n\n\n@dataclass\nclass ModelCardData:\n    model_name: str\n    version: str  # e.g., "fraud-champion-v1.0.0" per Section 17.3 versioning\n    training_data_snapshot_id: str\n    code_commit_hash: str\n\n    intended_use: str\n    out_of_scope_uses: list[str]\n\n    training_data_provenance: str\n    known_limitations: list[str]  # must include the Section 8/18 fairness-scope limitation\n\n    pr_auc_cv: float\n    pr_auc_cv_bootstrap_ci: tuple[float, float]\n    pr_auc_temporal_split: float\n    precision_at_threshold: float\n    recall_at_threshold: float\n    operating_threshold: float\n\n    external_benchmark_comparison: dict\n\n    monitoring_plan_reference: str = "See Master Playbook Section 9"\n    tier: int = 1\n\n    generated_at_utc: str = field(\n        default_factory=lambda: datetime.now(timezone.utc).isoformat()\n    )\n\n\ndef render_markdown(card: ModelCardData) -> str:\n    limitations = "\\n".join(f"- {item}" for item in card.known_limitations)\n    out_of_scope = "\\n".join(f"- {item}" for item in card.out_of_scope_uses)\n\n    return f"""# Model Card — {card.model_name}\n\n**Version:** {card.version}\n**Tier:** {card.tier} (per Master Playbook Section 3)\n**Training data snapshot:** {card.training_data_snapshot_id}\n**Code commit:** {card.code_commit_hash}\n**Generated:** {card.generated_at_utc}\n\n## Intended Use\n{card.intended_use}\n\n## Out-of-Scope Uses\n{out_of_scope}\n\n## Training Data Provenance\n{card.training_data_provenance}\n\n## Performance (real, measured)\n| Metric | Value |\n|---|---|\n| CV PR-AUC | {card.pr_auc_cv:.4f} |\n| CV PR-AUC 95% bootstrap CI | [{card.pr_auc_cv_bootstrap_ci[0]:.4f}, {card.pr_auc_cv_bootstrap_ci[1]:.4f}] |\n| Temporal-split PR-AUC | {card.pr_auc_temporal_split:.4f} |\n| Precision @ operating threshold ({card.operating_threshold:.4f}) | {card.precision_at_threshold:.4f} |\n| Recall @ operating threshold | {card.recall_at_threshold:.4f} |\n\n## External Benchmark Comparison (Section 19.4)\n{card.external_benchmark_comparison}\n\n## Known Limitations\n{limitations}\n\n## Monitoring Plan\n{card.monitoring_plan_reference}\n"""\n\n\ndef render_html(card: ModelCardData) -> str:\n    md = render_markdown(card)\n    # Minimal, dependency-free markdown-to-HTML for the specific structure\n    # this card always produces (headers, a table, bullet lists).\n    lines = md.splitlines()\n    html = ["<div class=\'model-card\'>"]\n    in_table = False\n    for line in lines:\n        if line.startswith("# "):\n            html.append(f"<h1>{line[2:]}</h1>")\n        elif line.startswith("## "):\n            html.append(f"<h2>{line[3:]}</h2>")\n        elif line.startswith("|"):\n            if "---" in line:\n                continue  # skip the markdown header-separator row\n            cells = [c.strip() for c in line.strip("|").split("|")]\n            if not in_table:\n                html.append("<table>")\n                in_table = True\n            html.append("<tr>" + "".join(f"<td>{c}</td>" for c in cells) + "</tr>")\n        else:\n            if in_table:\n                html.append("</table>")\n                in_table = False\n            if line.startswith("- "):\n                html.append(f"<li>{line[2:]}</li>")\n            elif line.strip():\n                html.append(f"<p>{line}</p>")\n    if in_table:\n        html.append("</table>")\n    html.append("</div>")\n    return "\\n".join(html)\n',
}
for _fname, _src in _MODULE_SOURCES.items():
    (_HERE / _fname).write_text(_src, encoding="utf-8")
for _modname in ("two_gate_validation", "drift_monitoring", "adversarial_robustness", "model_card_generator"):
    sys.modules.pop(_modname, None)

import two_gate_validation as tgv
import drift_monitoring as dm
import adversarial_robustness as adv
import model_card_generator as mcg

print("Self-installed local modules (UTF-8):", ", ".join(_MODULE_SOURCES))

##############################################################################
# LOAD NOTEBOOK 01's REAL OUTPUTS -- no retraining. Reuses the real,
# already cross-validated champion model and its real measured metrics.
##############################################################################
with open(os.path.join(NB1_RESULTS_DIR, "nb1_final_results.json"), encoding="utf-8") as f:
    nb1_results = json.load(f)

with open(os.path.join(NB1_RESULTS_DIR, "champion_model.pkl"), "rb") as f:
    champion_model = pickle.load(f)

FEATURE_COLS = list(champion_model.feature_names_)  # real, read from the trained model itself
CHOSEN_THRESHOLD = nb1_results["threshold_result"]["threshold"]
print(f"Loaded champion model ({nb1_results['champion_name']}), "
      f"{len(FEATURE_COLS)} features, operating threshold {CHOSEN_THRESHOLD:.4f}")

##############################################################################
# DATA_PATH resolution + Polars-accelerated load (Notebook 01's fixes reused
# unchanged: real dataset location checked first; explicit schema_overrides
# so the one row with Time written as scientific notation ("1.00E+05")
# doesn't break Polars' sample-based type inference).
##############################################################################
DATA_PATH = None
for _cand in [
    r"C:\Users\rnand\Downloads\creditcard.csv\creditcard.csv",  # your real dataset location -- checked first
    "creditcard.csv",
    os.path.join("data", "raw", "creditcard.csv"),
    os.path.join("..", "data", "raw", "creditcard.csv"),
    os.path.join("..", "creditcard.csv"),
]:
    if os.path.exists(_cand):
        DATA_PATH = _cand
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv not found. Place it next to this notebook, or at "
        "data/raw/creditcard.csv relative to the repo root."
    )
print("Using DATA_PATH:", DATA_PATH)

_SCHEMA_OVERRIDES = {"Time": pl.Float64, "Amount": pl.Float64, "Class": pl.Int64}
for _i in range(1, 29):
    _SCHEMA_OVERRIDES[f"V{_i}"] = pl.Float64

_t0 = time.time()
df = pl.read_csv(DATA_PATH, schema_overrides=_SCHEMA_OVERRIDES).to_pandas()
print(f"Loaded {len(df):,} rows x {len(df.columns)} cols via Polars in {time.time()-_t0:.3f}s")
_ram_after_load = psutil.virtual_memory()
print(f"RAM after load: {_ram_after_load.percent:.1f}% used ({_ram_after_load.used/1e9:.2f} GB / {_ram_after_load.total/1e9:.2f} GB)")

def score_fn(X_):
    """Real scoring function -- the champion model's own predict_proba, on
    its own real feature columns (V1-V28, Amount). Amount/Time perturbation
    columns can be present on the frame passed in; only FEATURE_COLS are
    ever handed to the model."""
    return champion_model.predict_proba(X_[FEATURE_COLS])[:, 1]

y_scores = score_fn(df)
y_pred = (y_scores >= CHOSEN_THRESHOLD).astype(int)
print(f"Scored {len(df):,} real rows. Flagged {int(y_pred.sum()):,} as fraud at threshold {CHOSEN_THRESHOLD:.4f}.")

# Real EUR/USD reference rate (ECB, 2026-09-11 fixing) -- same source used
# throughout Notebook 01.
EUR_TO_USD = 1.1592

# ============================================================
# GATE 1 -- Structural Integrity (Master Playbook Section 7)
# ============================================================
gate1 = tgv.run_gate1_structural_checks(
    df, required_columns=FEATURE_COLS + ["Time", "Class"], label_column="Class"
)
print("=" * 70)
print("GATE 1 -- STRUCTURAL INTEGRITY")
print("=" * 70)
for _k, _v in gate1.checks.items():
    print(f"  {_k}: {_v}")
print(f"GATE 1 ALL PASSED: {gate1.all_passed}")

# ============================================================
# GATE 2 -- Statistical Robustness / Concentration / Stress-Test
# ============================================================
concentration = tgv.concentration_report_by_amount_and_time(
    df.assign(_pred=y_pred), amount_col="Amount", time_col="Time",
    y_true_col="Class", y_pred_col="_pred",
)
concentration.to_csv(os.path.join(RESULTS_DIR, "concentration_report.csv"), index=False, encoding="utf-8")
print(f"Concentration report -- {len(concentration)} amount-band x hour-of-day segments "
      f"(saved to {os.path.join(RESULTS_DIR, 'concentration_report.csv')})")
display(concentration.sort_values("false_negative_rate", ascending=False).head(10))

base_fraud_rate = nb1_results["dataset"]["fraud_rate"]
stress = tgv.stress_test_scenario(
    df, amount_col="Amount",
    scenario_volume_multiplier=2.0,       # ASSUMPTION: documented hypothetical stress shock
    scenario_fraud_rate_multiplier=3.0,   # ASSUMPTION: documented hypothetical stress shock
    base_fraud_rate=base_fraud_rate,
)
stress["projected_fraud_loss_usd"] = stress["projected_fraud_loss_usd_or_eur"] * EUR_TO_USD
stress["eur_to_usd_rate"] = EUR_TO_USD
print("=" * 70)
print("GATE 2 -- STRESS-TEST SCENARIO (both multipliers are stated ASSUMPTIONs, not fitted)")
print("=" * 70)
for _k, _v in stress.items():
    print(f"  {_k}: {_v}")

gate2 = tgv.Gate2Result(
    cv_auc_scores=nb1_results["stage_b"]["CatBoost"]["fold_pr_auc"],
    cv_pr_auc_scores=nb1_results["stage_b"]["CatBoost"]["fold_pr_auc"],
    concentration_report=concentration,
    stress_test_report=stress,
)
print(f"GATE 2 CV stability OK (fold PR-AUC std <= 0.03): {gate2.cv_stability_ok}")

# ============================================================
# DRIFT MONITORING BASELINE (Master Playbook Section 9)
# Real limitation, disclosed: only a ~48-hour window exists in this dataset,
# so a genuine multi-period drift run isn't possible yet. This uses the
# module's own documented early-vs-late proxy as the first real drift-history
# data point, not a substitute for real production monitoring.
# ============================================================
early, late = dm.early_vs_late_window_proxy(df, time_col="Time")

# Top features by the champion model's own real (PredictionValuesChange)
# feature importance -- a fast, real substitute for re-running full SHAP
# here; Notebook 01's SHAP values were rendered to a figure, not persisted
# as numbers, so this reads real importances straight from the real model
# rather than re-deriving or guessing them.
_importances = champion_model.get_feature_importance()
_top_idx = np.argsort(_importances)[::-1][:10]
top_features = [FEATURE_COLS[i] for i in _top_idx]
print("Top-10 features by real CatBoost feature importance:", top_features)

from sklearn.metrics import average_precision_score
pr_auc_early = average_precision_score(early["Class"], score_fn(early))
pr_auc_late = average_precision_score(late["Class"], score_fn(late))

drift_report = dm.compute_drift_report(
    training_features=early[top_features], current_features=late[top_features],
    training_scores=score_fn(early), current_scores=score_fn(late),
    pr_auc_baseline=pr_auc_early, pr_auc_current=pr_auc_late,
    top_features=top_features, tier=1,
)
print("=" * 70)
print("DRIFT MONITORING -- EARLY vs LATE WINDOW PROXY (real, computed)")
print("=" * 70)
print(f"  Early window: {len(early):,} rows | Late window: {len(late):,} rows")
print(f"  PR-AUC early: {pr_auc_early:.4f} | PR-AUC late: {pr_auc_late:.4f} | drop: {drift_report.pr_auc_drop:.4f}")
print(f"  Score KS statistic: {drift_report.score_ks_statistic:.4f} (p={drift_report.score_ks_pvalue:.4g})")
print(f"  Feature PSI (top-10 by importance): {drift_report.feature_psi}")
print(f"  Alerts (Tier {drift_report.tier} thresholds): {drift_report.alerts}")
print(f"  ANY ALERT: {drift_report.any_alert}")

# ============================================================
# ADVERSARIAL ROBUSTNESS TEST (Master Playbook Section 19)
# ============================================================
X_fraud = df[df["Class"] == 1].copy()
sens_results = adv.perturbation_sensitivity_test(
    score_fn=score_fn, X_fraud=X_fraud, threshold=CHOSEN_THRESHOLD,
)
boundary_result = adv.boundary_search_attack(
    score_fn=score_fn, X_fraud=X_fraud, threshold=CHOSEN_THRESHOLD,
)
adv.print_report(sens_results, boundary_result)

# ============================================================
# MODEL CARD (Master Playbook Section 11)
# ============================================================
try:
    _git = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
                           capture_output=True, text=True, check=True)
    code_commit_hash = _git.stdout.strip()
except Exception:
    code_commit_hash = "unavailable (git not found, or not a git repo at run time)"

card = mcg.ModelCardData(
    model_name="Fraud Detection Champion (CatBoost)",
    version="fraud-champion-v1.0.0",
    training_data_snapshot_id=f"creditcard.csv, {len(df):,} rows, RANDOM_SEED={RANDOM_SEED}",
    code_commit_hash=code_commit_hash,
    intended_use="Real-time, transaction-level fraud-probability scoring for card-not-present "
                 "and card-present transactions, to prioritize which transactions a fraud-ops "
                 "team reviews or holds, at the cost-optimal decision threshold computed in "
                 "Notebook 01.",
    out_of_scope_uses=[
        "Any automated account-closure or customer-blocking decision without human review.",
        "Any use as a credit, lending, or account-opening decision -- this model is trained "
        "only on real-time transaction fraud, not creditworthiness.",
        "Any demographic or fairness-protected-class decisioning -- the training features are "
        "anonymized PCA components (V1-V28); no demographic attributes are available or used.",
    ],
    training_data_provenance="Kaggle mlg-ulb/creditcardfraud (Worldline / Universite Libre de "
                              "Bruxelles), real anonymized European cardholder transactions, "
                              "September 2013, ~48-hour window, 284,807 transactions, 492 fraud "
                              "(0.173%).",
    known_limitations=[
        "Anonymized PCA features (V1-V28) make demographic fairness auditing impossible on this "
        "dataset -- disclosed per Section 8/18, not silently omitted.",
        "Only a ~48-hour data window exists; the drift-monitoring baseline above uses an "
        "early-vs-late proxy, not genuine multi-period production history (Section 9).",
        f"Adversarial robustness (Section 19): {boundary_result.n_evadable_within_budget} of "
        f"{boundary_result.n_fraud_cases_tested} currently-caught fraud cases "
        f"({boundary_result.evasion_rate_within_budget:.2%}) were evadable within a "
        f"{boundary_result.max_amount_change_pct_budget:.0%} amount-cut budget -- see the "
        "adversarial robustness section above for the real, measured numbers.",
        "Gate 2 stress-test multipliers (2x volume, 3x fraud rate) are stated ASSUMPTIONs for "
        "a documented hypothetical shock, not a fitted or historically-calibrated scenario.",
    ],
    pr_auc_cv=nb1_results["stage_b"]["CatBoost"]["mean_pr_auc"],
    pr_auc_cv_bootstrap_ci=tuple(nb1_results["stage_b"]["CatBoost"]["bootstrap_ci"]),
    pr_auc_temporal_split=nb1_results["temporal_pr_auc"],
    precision_at_threshold=nb1_results["real_precision"],
    recall_at_threshold=nb1_results["real_recall"],
    operating_threshold=CHOSEN_THRESHOLD,
    external_benchmark_comparison=nb1_results["benchmark_check"],
    tier=1,
)
card_md = mcg.render_markdown(card)
card_html = mcg.render_html(card)

with open(os.path.join(RESULTS_DIR, "model_card.md"), "w", encoding="utf-8") as f:
    f.write(card_md)
with open(os.path.join(RESULTS_DIR, "model_card.html"), "w", encoding="utf-8") as f:
    f.write(card_html)
print(card_md)

# ============================================================
# DEPLOYMENT READINESS CHECK (Master Playbook Section 17.4)
# Per the standing execution-boundary rule, this notebook does not spin up
# a live API server -- deployment/app.py is a separate, already-written
# FastAPI service. This section only confirms the real artifacts it needs
# actually exist and are loadable, and prints how to run it for real.
# ============================================================
_app_path = os.path.join(REPO_ROOT, "deployment", "app.py")
_model_path = os.path.join(NB1_RESULTS_DIR, "champion_model.pkl")
print("=" * 70)
print("DEPLOYMENT READINESS")
print("=" * 70)
print(f"  deployment/app.py present: {os.path.exists(_app_path)}")
print(f"  Champion model artifact present and loadable: {os.path.exists(_model_path)} "
      f"(loaded above as champion_model)")
print(f"  To run the real API: MODEL_PATH={_model_path} MODEL_VERSION=v1.0.0 "
      f"DECISION_THRESHOLD={CHOSEN_THRESHOLD:.4f} uvicorn app:app --host 0.0.0.0 --port 8000")
print("  (run from the deployment/ folder, per deployment/app.py's own docstring)")

# ============================================================
# SAVE NOTEBOOK 02 RESULTS -- reports/nb2_results/nb2_validation_report.json
# ============================================================
nb2_report = {
    "run_metadata": {"random_seed": RANDOM_SEED, "n_rows": len(df), "n_threads": _N_THREADS,
                     "code_commit_hash": code_commit_hash},
    "gate1_structural_checks": {k: (bool(v) if isinstance(v, np.bool_) else v) for k, v in gate1.checks.items()},
    "gate1_all_passed": gate1.all_passed,
    "gate2_cv_stability_ok": gate2.cv_stability_ok,
    "gate2_stress_test": stress,
    "drift_monitoring": {
        "early_window_rows": int(len(early)), "late_window_rows": int(len(late)),
        "pr_auc_early": pr_auc_early, "pr_auc_late": pr_auc_late,
        "pr_auc_drop": drift_report.pr_auc_drop,
        "score_ks_statistic": drift_report.score_ks_statistic,
        "score_ks_pvalue": drift_report.score_ks_pvalue,
        "feature_psi": drift_report.feature_psi,
        "alerts": drift_report.alerts, "any_alert": drift_report.any_alert,
        "top_features_by_importance": top_features,
    },
    "adversarial_robustness": {
        "perturbation_sensitivity": [vars(r) for r in sens_results],
        "boundary_search": {
            "n_fraud_cases_tested": boundary_result.n_fraud_cases_tested,
            "n_evadable_within_budget": boundary_result.n_evadable_within_budget,
            "evasion_rate_within_budget": boundary_result.evasion_rate_within_budget,
            "median_amount_change_pct_for_evasion": boundary_result.median_amount_change_pct_for_evasion,
            "step_pct": boundary_result.step_pct,
            "max_amount_change_pct_budget": boundary_result.max_amount_change_pct_budget,
        },
    },
    "eur_to_usd_rate": EUR_TO_USD,
    "eur_to_usd_source": "ECB euro foreign exchange reference rate, 2026-09-11 fixing "
                          "(https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-usd.en.html)",
}
with open(os.path.join(RESULTS_DIR, "nb2_validation_report.json"), "w", encoding="utf-8") as f:
    json.dump(nb2_report, f, indent=2, default=str)

_ram_end = psutil.virtual_memory()
print("=" * 70)
print(f"RAM at finish: {_ram_end.percent:.1f}% used ({_ram_end.used/1e9:.2f} GB / {_ram_end.total/1e9:.2f} GB)")
print(f"Notebook 02 complete. Results written to: {RESULTS_DIR}")
print(f"  - nb2_validation_report.json")
print(f"  - concentration_report.csv")
print(f"  - model_card.md")
print(f"  - model_card.html")
